## C5_02 — Construirea vector store-ului pentru o bulă
În acest notebook construim un vector store FAISS pentru o singură bulă / un singur agent.
Fiecare student lucrează pe bula lui. Scopul este să vedem clar cum textele curățate devin embeddings, apoi index FAISS.
Mai târziu, aceeași logică va fi pusă într-un script `.py` care rulează automat pentru toate bulele.

## 0. Setup

In [1]:
from pathlib import Path
import os, pickle
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

while not Path("data/bubbles").exists():
    os.chdir("..")

BUBBLES_DIR = Path("data/bubbles")
VECTOR_DIR = Path("assets/vectorstores")
VECTOR_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"

C:\Users\vitok\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\vitok\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


## 1. Aleg bula mea
Alege fișierul `.jsonl` al bulei tale.
Acest fișier a fost creat în etapa anterioară, după verificarea manuală a textelor.

In [3]:
MY_BUBBLE_FILE = "anti_suveranist.jsonl" 

bubble_path = BUBBLES_DIR / MY_BUBBLE_FILE
slug = bubble_path.stem

df_bubble = pd.read_json(bubble_path, lines=True)

print("Bula:", slug)
print("Texte:", len(df_bubble))

df_bubble[["id", "agent", "text"]].head()

Bula: anti_suveranist
Texte: 50


,id,agent,text
0,yt_Tx8GhU2LeyI_UgwoWOyzF2UbPYnguUB4AaABAg,Anti-suveranist,Am toată încrederea că oameni ( de bine ) ca :...
1,yt_6_Hc2S02Duw_Ugytw6-BDQ2pA_Zi-TB4AaABAg,Anti-suveranist,Apropo de avalansa de troli ce se devarsa si a...
2,yt_im3QoqSgfDo_UgwOL2VI3_toiLZSDqh4AaABAg,Anti-suveranist,"Eu am vorbit cu susținători de ai lui CG, îs d..."
3,yt_6hgc90sLVFw_UgzKEAwJ4lrjMa0IzeB4AaABAg,Anti-suveranist,Este evident ca fortele din spatele lui simion...
4,yt_N5Wr2OHINDo_Ugw9Gyk4Ehe3UofP07V4AaABAg,Anti-suveranist,Adică UDMR a fost cu Psd tot timpul la guverna...


## 2. Pregătim textele
Pentru FAISS avem nevoie de o listă simplă de texte.
Metadata rămâne separat, ca să putem lega fiecare vector de textul original.

In [4]:
texts = df_bubble["text"].fillna("").tolist()
metadata = df_bubble.to_dict(orient="records")

print("Primul text:")
print(texts[0][:500])

Primul text:
Am toată încrederea că oameni ( de bine ) ca : G Simion , Călinge , dna Găurilă ...... vor avea mare grijă să pună fie piedici , fie bețe-n roate astfel încât să rămanem sub tutela cremlinului


## 3. Generăm embeddings
Un embedding este o reprezentare vectorială a textului: texte apropiate ca sens primesc vectori apropiați în spațiul semantic.
Folosim un model multilingv, deoarece corpusul este în limba română.
Normalizăm vectorii la lungime 1, astfel încât produsul scalar din FAISS să funcționeze ca similaritate cosinus.

In [5]:
model = SentenceTransformer(MODEL_NAME)
embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")
print("Număr texte:", len(texts))
print("Dimensiune embeddings:", embeddings.shape)

Batches: 100%|██████████| 2/2 [00:03<00:00,  1.69s/it]

Număr texte: 50
Dimensiune embeddings: (50, 384)


### Verificare rapidă
Răspunde în 1–2 propoziții în notebook:
- Câte texte are bula ta?
- Câți vectori au fost generați?
- Ce înseamnă a doua valoare din `embeddings.shape`?

In [8]:
# TODO student:
# Bula mea are 50 texte.
# Au fost generați 50 vectori.
# A doua valoare din embeddings.shape reprezintă dimensiunea vectorilor, 384 fiind pentru acest model.

## 4. Construim indexul FAISS
FAISS este biblioteca care caută rapid vectori apropiați.
Indexul nu păstrează textele originale. El păstrează doar reprezentările vectoriale.
De aceea salvăm două lucruri:
- `index.faiss` = indexul vectorial;
- `index.pkl` = textele originale și metadatele.

In [9]:
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
out_dir = VECTOR_DIR / slug
out_dir.mkdir(parents=True, exist_ok=True)
faiss.write_index(index, str(out_dir / "index.faiss"))
with open(out_dir / "index.pkl", "wb") as f:
    pickle.dump(metadata, f)
print("Salvat în:", out_dir)
print("Vectori în index:", index.ntotal)

Salvat în: assets\vectorstores\anti_suveranist
Vectori în index: 50


## 5. Verificăm fișierele create
Dacă totul a mers corect, bula ta are acum un folder propriu în `assets/vectorstores/`.
Acest folder trebuie să conțină `index.faiss` și `index.pkl`.

In [13]:
# TODO student:
# index.faiss există: Da
# index.pkl există: Da
# index.ntotal este egal cu numărul de texte: Da (50)

## Ce am construit?
Am transformat textele curate ale unei bule într-un index vectorial local.
Acest index nu generează răspunsuri. El doar permite căutarea semantică.
În următorul continuare vom testa dacă, pentru o întrebare, FAISS returnează texte relevante.

## 6. Testăm retrieval-ul
Acum simulăm logica aplicației.
- Utilizatorul introduce o știre sau o afirmație politică.
- Retriever-ul caută în memoria bulei cele mai asemănătoare texte.
- Nu generăm încă un răspuns cu LLM. Doar verificăm ce exemple sunt recuperate.

In [17]:
# Text nou introdus în aplicație

input_text = "Cum încearcă populiștii și suveraniștii să destabilizeze instituțiile democratice?"

In [18]:
# Transformăm textul nou în embedding

query_vector = model.encode(
    [input_text],
    normalize_embeddings=True
).astype("float32")

In [19]:
query_vector

array([[-3.55359004e-03,  2.69677956e-02, -1.61741450e-02,
        -9.86449793e-03,  1.03532583e-01, -3.91791165e-02,
        -6.22690506e-02, -3.65001969e-02, -2.54012235e-02,
         4.93087023e-02,  2.31351405e-02,  4.91065383e-02,
         5.36308028e-02, -1.80597510e-02,  5.26209585e-02,
         1.20889600e-02, -8.26486573e-02,  2.01311745e-02,
        -2.45230254e-02,  1.24805689e-01,  4.50084656e-02,
        -7.04330802e-02, -2.43773684e-02,  3.85253020e-02,
         6.13580830e-02,  4.03569862e-02,  3.51891927e-02,
         5.02344500e-03, -1.35792503e-02, -5.73754720e-02,
         1.03062158e-02, -1.05003186e-01, -2.89593209e-02,
         5.04216366e-02,  9.35652629e-02, -5.49700297e-02,
         6.94883019e-02,  2.54024062e-02,  4.06283252e-02,
        -4.14202325e-02,  7.05028176e-02, -7.14668930e-02,
         2.06541009e-02, -1.72779467e-02,  1.99324824e-02,
        -4.29194160e-02,  2.89116204e-02,  3.70793231e-02,
        -3.64848338e-02, -7.62828588e-02,  6.98721558e-0

In [20]:
# Căutăm cele mai apropiate 5 texte din bula noastră

scores, results = index.search(query_vector, k=5)

for rank, pos in enumerate(results[0], start=1):
    row = metadata[pos]
    
    print(f"\nRezultat {rank}")
    print("Scor:", round(float(scores[0][rank-1]), 3))
    print("Text:", row["text"][:500])


Rezultat 1
Scor: 0.461
Text: o analiză detaliată, dar trebuie să fim foarte atenți la cum abordăm subiectele politice, mai ales când e vorba de partide și conflicte. Este important să discutăm într-un mod respectuos și informat, având în vedere că astfel de subiecte pot fi foarte sensibile și pot avea un impact puternic asupra opiniei publice. Cum poate opoziția din partidu AUR să încerce să erodeze democrația? Manipularea narativului Opoziția poate încerca să submineze încrederea cetățenilor în instituțiile democratice, pr

Rezultat 2
Scor: 0.413
Text: Puteti sa "cautati"cauzele pentru care Simion a plimbat marea masa a protestatorilor pe o ruta in afara zonei centrale, guvernamentale - pana la Cotroceni unde au fost doar blindate de jamdarmi- pana a epuizat pe toti cei ce iesisera la protest. Pur si simplu dubioasa "decizia"lui Simion. Si de atunci ...romanii nu mai ies la proteste, le-a stins orice speranta. Dubios dl Simion, f dubios.

Rezultat 3
Scor: 0.412
Text: Si ce daca a fos

### TODO
Schimbă `input_text` cu o afirmație potrivită pentru agentul tău.
Rulează căutarea.
Notează:
- câte rezultate din 5 sunt relevante;
### Sunt relevante 4 din 5, aratand o atitudine critica, suspicioasa la adresa liderilor suveranisti. 
- dacă textele recuperate exprimă vocea agentului;
### Rezultatele 2 si 5 exprima cat de cat vocea agentului; il critica pe Simion, il numesc "dubios" si sugereaza ca saboteaza protestele din umbra sau ca face un joc dublu in spatele mastii de suveranist.
- dacă ai observat un text slab care ar trebui eliminat.
### Din punct de vedere tehnic, sunt doua texte care ar trebui eliminate: rezultatul 1 si 5, textele sunt foarte slab procesate.